In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [15]:
# Create binary target using median CTR

median_ctr = df["ctr"].median()

df["high_performance"] = (df["ctr"] > median_ctr).astype(int)

df["high_performance"].value_counts()

high_performance
0    15224
1    14776
Name: count, dtype: int64

In [16]:
# Select numerical features

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engagement_rate",
    "avg_position",
    "content_age_days"
]

X = df[features].copy()
y = df["high_performance"]

# Fill missing values using the median
X = X.fillna(X.median())

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(24000, 13)
(6000, 13)


In [18]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_pred))
print("Baseline F1 Score:", f1_score(y_test, baseline_pred))

Baseline Accuracy: 0.5075
Baseline F1 Score: 0.0


In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Model Accuracy:", accuracy_score(y_test, pred))
print("Model F1 Score:", f1_score(y_test, pred))

Model Accuracy: 0.992
Model F1 Score: 0.9918228279386712


c:\Users\adity\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [20]:
X.isnull().sum()

search_volume       0
competition         0
cpc                 0
word_count          0
char_count          0
impressions_90d     0
clicks_90d          0
pageviews_90d       0
sessions_90d        0
users_90d           0
engagement_rate     0
avg_position        0
content_age_days    0
dtype: int64

In [21]:
comparison = pd.DataFrame({
    "Model": ["Baseline (Dummy)", "Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, pred)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred),
        f1_score(y_test, pred)
    ]
})

comparison

,Model,Accuracy,F1 Score
0,Baseline (Dummy),0.5075,0.000000
1,Logistic Regression,0.9920,0.991823


In [22]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

importance["Absolute Importance"] = importance["Coefficient"].abs()

importance.sort_values(
    by="Absolute Importance",
    ascending=False
).head(10)

,Feature,Coefficient,Absolute Importance
6,clicks_90d,22.207781,22.207781
9,users_90d,-0.336252,0.336252
8,sessions_90d,0.328396,0.328396
2,cpc,-0.193004,0.193004
7,pageviews_90d,0.020696,0.020696
5,impressions_90d,-0.016376,0.016376
10,engagement_rate,0.007529,0.007529
12,content_age_days,-0.005469,0.005469
1,competition,0.002122,0.002122
11,avg_position,0.002042,0.002042


# 1. Method Choice

I selected **Logistic Regression** because the task is a binary classification problem. It is simple, interpretable, and provides a strong baseline for predicting whether a content page is high-performing. It also allows us to understand how each feature influences the prediction.

# 2. Split Design

The dataset was divided into:

- Training Set: 80%
- Testing Set: 20%

A fixed random state (42) and stratified sampling were used to ensure reproducible and balanced evaluation.

# 3. Train + Compare Against Baseline

A Dummy Classifier was used as the Week 4 baseline.

The Logistic Regression model was trained on the same training data and evaluated using the same test split.

The comparison table shows that the trained model performs better than the baseline on the chosen evaluation metrics.

# 4. Errors and Interpretation

The model may incorrectly classify pages whose performance is close to the decision boundary.

Feature coefficients indicate which variables have the strongest influence on predicting high-performing content. This helps explain model behavior and supports data-driven SEO decisions.

# 5. Self Check

- ✅ Method choice explained
- ✅ Valid train/test split used
- ✅ Compared against Week 4 baseline
- ✅ Reported Accuracy and F1 Score
- ✅ Interpreted important features
- ✅ Explained model limitations
